## ASSIGNMENT 1 : Risk Measure Analysis (Market & Credit Risk)
## OC SEITSANG
## 221011692

#### LIBRARIES 

In [56]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
import cvxpy as cp
from scipy.stats import jarque_bera, shapiro, skew, kurtosis
from scipy.stats import norm

In [24]:
license_path= r"C:\Program Files\Mosek\mosek.lic"

In [25]:

x = cp.Variable()
prob = cp.Problem(cp.Minimize((x - 2)**2))
prob.solve(solver=cp.MOSEK)

print("Optimal x:", x.value)

Optimal x: 2.0


## importing the data from Yahoo Finance

In [33]:

# Asset selection
stocks = ["SBK.JO", "NPN.JO", "MTN.JO", "SHP.JO", "SOL.JO"]
bonds = ["LQD", "HYG"]   # corporate-bond proxies
tickers = stocks + bonds

# Download adjusted close prices
prices = yf.download(tickers,start="2020-01-01",end="2025-01-01",auto_adjust=True,progress=False)["Close"]


In [68]:

# Check the result
print(prices.head())
print(prices.tail())
print(prices.isna().sum()) # checking NA's 

Ticker            HYG         LQD       MTN.JO        NPN.JO        SBK.JO  \
Date                                                                         
2020-01-02  63.542240  103.002693  8349.507812  46741.820312  16816.675781   
2020-01-03  63.477444  103.356133  8107.955566  47071.761719  16661.263672   
2020-01-06  63.412712  103.010704  8024.111816  46867.792969  16277.708984   
2020-01-07  63.355186  102.745628  8025.109375  47153.746094  16290.660156   
2020-01-08  63.412712  102.601067  7946.256348  46491.863281  16374.341797   

Ticker            SHP.JO        SOL.JO  
Date                                    
2020-01-02  12503.827148  30347.042969  
2020-01-03  12351.085938  31161.023438  
2020-01-06  11977.717773  31795.232422  
2020-01-07  12179.376953  31495.607422  
2020-01-08  12208.329102  31103.095703  
Ticker            HYG         LQD       MTN.JO        NPN.JO        SBK.JO  \
Date                                                                         
2024-12-23

## clean the data
#### Drop rows with missing values across selected assets

In [69]:
prices = prices.dropna()

print("Shape of cleaned price data:", prices.shape)
print(prices.head())

# Check the result
print(prices.head())
print(prices.tail())
print(prices.isna().sum())

Shape of cleaned price data: (1216, 7)
Ticker            HYG         LQD       MTN.JO        NPN.JO        SBK.JO  \
Date                                                                         
2020-01-02  63.542240  103.002693  8349.507812  46741.820312  16816.675781   
2020-01-03  63.477444  103.356133  8107.955566  47071.761719  16661.263672   
2020-01-06  63.412712  103.010704  8024.111816  46867.792969  16277.708984   
2020-01-07  63.355186  102.745628  8025.109375  47153.746094  16290.660156   
2020-01-08  63.412712  102.601067  7946.256348  46491.863281  16374.341797   

Ticker            SHP.JO        SOL.JO  
Date                                    
2020-01-02  12503.827148  30347.042969  
2020-01-03  12351.085938  31161.023438  
2020-01-06  11977.717773  31795.232422  
2020-01-07  12179.376953  31495.607422  
2020-01-08  12208.329102  31103.095703  
Ticker            HYG         LQD       MTN.JO        NPN.JO        SBK.JO  \
Date                                             

## convert prices to returns

In [70]:
returns = prices.pct_change().dropna()
print(returns.head())
print("Shape of returns data:", returns.shape)

Ticker           HYG       LQD    MTN.JO    NPN.JO    SBK.JO    SHP.JO  \
Date                                                                     
2020-01-03 -0.001020  0.003431 -0.028930  0.007059 -0.009242 -0.012216   
2020-01-06 -0.001020 -0.003342 -0.010341 -0.004333 -0.023021 -0.030230   
2020-01-07 -0.000907 -0.002573  0.000124  0.006101  0.000796  0.016836   
2020-01-08  0.000908 -0.001407 -0.009826 -0.014037  0.005137  0.002377   
2020-01-09  0.002610  0.004776 -0.001884  0.006963 -0.001034 -0.005479   

Ticker        SOL.JO  
Date                  
2020-01-03  0.026822  
2020-01-06  0.020353  
2020-01-07 -0.009424  
2020-01-08 -0.012462  
2020-01-09 -0.027166  
Shape of returns data: (1215, 7)


## Building a portfolio

In [71]:
# Defining the weights
n_assets = len(returns.columns)
weights = np.ones(n_assets) / n_assets

In [72]:
#portfolio Returns

In [39]:
portfolio_returns = returns @ weights

## RISK MEASURE 

#### Test for Normality
The Jarque-Bera and Shapiro-Wilk tests are used to assess whether the portfolio returns follow a normal distribution. Skewness and kurtosis are also computed to evaluate asymmetry and tail behavior.

In [76]:

# Portfolio returns
r = portfolio_returns

# Jarque-Bera test
jb_stat, jb_p = jarque_bera(r)

# Shapiro-Wilk test
shapiro_stat, shapiro_p = shapiro(r.sample(500))  # sample if large

# Skewness & Kurtosis
sk = skew(r)
kt = kurtosis(r)

print("Jarque-Bera p-value:", jb_p)
print("Shapiro p-value:", shapiro_p)
print("Skewness:", sk)
print("Kurtosis:", kt)




Jarque-Bera p-value: 0.0
Shapiro p-value: 5.88059397871805e-19
Skewness: -0.3660601127732308
Kurtosis: 12.150313808257284


### interpretation
Both tests strongly reject normality (p-values = 0), indicating that the portfolio returns are not normally distributed.

The skewness of -0.366 shows slight negative asymmetry, meaning extreme negative returns are more likely than extreme positive ones.

The kurtosis of 12.15 is significantly higher than the normal benchmark of 3, indicating fat tails and a high probability of extreme returns.

Overall, the portfolio exhibits non-normality, downside risk, and heavy tails, which are typical characteristics of financial return series.

## Risk Measures: VaR and CVaR
We calculate both historical and normal-based VaR and CVaR at the 95% level to compare risk under real data vs a normal assumption.

In [82]:

mu = r.mean()
sigma = r.std()

z = norm.ppf(0.05)

VaR_normal = -(mu + z * sigma)
CVaR_normal = -(mu - sigma * (norm.pdf(z) / alpha))

print("Normal CVaR (95%):", CVaR_normal)
print("Normal VaR (95%):", VaR_normal)

Normal CVaR (95%): 0.02819278931754963
Normal VaR (95%): 0.022398793659688304


In [ ]:
# C

In [40]:
daily_vol = portfolio_returns.std()
annual_vol = daily_vol * np.sqrt(252)

print("Daily Volatility:", daily_vol)
print("Annual Volatility:", annual_vol)

Daily Volatility: 0.013865904896842382
Annual Volatility: 0.22011441635950596


In [41]:
## 2. Var 95% 

In [42]:
alpha = 0.05

VaR_95 = -np.percentile(portfolio_returns, alpha * 100)
print("VaR 95%:", VaR_95)

VaR 95%: 0.018513676396723324


In [43]:
### CVar 

In [44]:
cutoff = np.percentile(portfolio_returns, alpha * 100)
CVaR_95 = -portfolio_returns[portfolio_returns <= cutoff].mean()
print("CVaR 95%:", CVaR_95)

CVaR 95%: 0.030375044830037726


In [84]:
# Results table
results = pd.DataFrame({
    "Measure": [
        "Daily Vol",
        "VaR 95% (historical)",
        "CVaR 95% (historical)",
        "VaR 95% (Normal)",
        "CVaR 95% (Normal)"
    ],
    "Value": [
        daily_vol,
        VaR_95,
        CVaR_95,
        VaR_normal,
        CVaR_normal
    ]
})
print(results)

                 Measure     Value
0              Daily Vol  0.013866
1   VaR 95% (historical)  0.018514
2  CVaR 95% (historical)  0.030375
3       VaR 95% (Normal)  0.022399
4      CVaR 95% (Normal)  0.028193


#### interpretation

The historical Value-at-Risk (VaR) of 1.85\% and Conditional Value-at-Risk (CVaR) of 3.04\% indicate that while typical losses are moderate, extreme losses can be significantly larger, highlighting the presence of tail risk. Although the normal VaR and CVaR are close to the historical estimates, they slightly underestimate extreme losses, and therefore the historical CVaR is the more reliable measure given that the returns are not normally distributed.

### CVAR Optimization
We optimise the portfolio weights by minimising CVaR, so the aim is to find the set of weights that gives the lowest expected loss in the worst 5% of cases

In [85]:
import cvxpy as cp

# Convert to numpy for optimisation later
R = returns.to_numpy()
T, n = R.shape

x_cvar = cp.Variable((n, 1))
t_cvar = cp.Variable((1, 1))
u_cvar = cp.Variable((T, 1))

constraints = [
    cp.sum(x_cvar) == 1,
    x_cvar >= 0,
    u_cvar >= -R @ x_cvar - t_cvar,
    u_cvar >= 0
]

risk_cvar = t_cvar + (1 / (alpha * T)) * cp.sum(u_cvar)

prob_cvar = cp.Problem(cp.Minimize(risk_cvar), constraints)
prob_cvar.solve(solver="MOSEK")

print("Optimal CVaR:", prob_cvar.value)

Optimal CVaR: 0.013740433305391011


### Interpretation

The optimal CVaR is about 1.37%, which means the chosen weights reduce the portfolio’s average extreme loss to this level. This suggests the optimised portfolio is quite effective at controlling downside tail risk.

# EVaR
We optimise the portfolio weights by minimising the Entropic Value-at-Risk (EVaR), so the goal is to find the set of weights that gives the lowest tail risk.

In [86]:
x_evar = cp.Variable((n, 1))
t_evar = cp.Variable((1, 1))
z_evar = cp.Variable((1, 1), nonneg=True)
u_evar = cp.Variable((T, 1))

ones = np.ones((T
                , 1))

constraints = [
    cp.sum(x_evar) == 1,
    x_evar >= 0,
    cp.sum(u_evar) <= z_evar,
    cp.ExpCone(-R @ x_evar - t_evar, ones @ z_evar, u_evar)
]

risk_evar = t_evar + z_evar * np.log(1 / (alpha * T))

prob_evar = cp.Problem(cp.Minimize(risk_evar), constraints)
prob_evar.solve(solver="MOSEK")

print("Optimal EVaR:", prob_evar.value)

Optimal EVaR: 0.028086055844292735


### interpretation

The optimal EVaR is about 2.81%, which is the minimum tail risk achieved by the chosen portfolio weights. In other words, these weights produce the safest portfolio under the EVaR criterion

## cDar DrowDown
We optimise the portfolio weights by minimising Conditional Drawdown-at-Risk (CDaR), which focuses on limiting the portfolio’s worst drawdowns over time.

In [88]:
x_cdar = cp.Variable((n, 1))
d = cp.Variable((T + 1, 1))
t = cp.Variable((1, 1))
u = cp.Variable((T, 1))

constraints = [
    cp.sum(x_cdar) == 1,
    x_cdar >= 0,
    d[1:] >= d[:-1] - R @ x_cdar,
    d[1:] >= 0,
    d[0] == 0,
    u >= d[1:] - t,
    u >= 0
]

risk_cdar = t + (1 / (alpha * T)) * cp.sum(u)

prob_cdar = cp.Problem(cp.Minimize(risk_cdar), constraints)
prob_cdar.solve(solver="MOSEK")

print("Optimal CDaR:", prob_cdar.value)

Optimal CDaR: 0.09868410942686945


## interpretation
The optimal CDaR is about 9.87%, meaning the portfolio’s average worst drawdowns are around this level under the optimal weights. This shows the portfolio can still experience fairly noticeable peak-to-trough losses, even after optimisation.

## Sumaary of Market risk measures 


In [90]:
# Results table
results = pd.DataFrame({
    "Measure": [
        "Daily Vol",
        "VaR 95% (historical)",
        "CVaR 95% (historical)",
        "VaR 95% (Normal)",
        "CVaR 95% (Normal)",
        "Optimal CVaR",
        "Optimal EVaR",
        "Optimal CDaR"
    ],
    "Value": [
        daily_vol,
        VaR_95,
        CVaR_95,
        VaR_normal,
        CVaR_normal,
        prob_cvar.value,
        prob_evar.value,
        prob_cdar.value
    ]
})

results

,Measure,Value
0,Daily Vol,0.013866
1,VaR 95% (historical),0.018514
2,CVaR 95% (historical),0.030375
3,VaR 95% (Normal),0.022399
4,CVaR 95% (Normal),0.028193
5,Optimal CVaR,0.013740
6,Optimal EVaR,0.028086
7,Optimal CDaR,0.098684


### Interpretation of Results

The portfolio has moderate daily volatility (about 1.39%), but the risk measures show that losses can become much larger in bad market conditions. The historical CVaR of 3.04% confirms that when things go wrong, losses can quickly escalate beyond the usual day-to-day movements.

The normal-based VaR and CVaR are fairly close to the historical ones, but since the returns are not normally distributed, they don’t fully capture the true risk, especially in the tails.

After optimisation, the CVaR drops to 1.37%, which is a big improvement and shows that the portfolio can be adjusted to significantly reduce extreme losses. The EVaR is higher at 2.81%, giving a more conservative estimate of risk, while the CDaR is much larger at 9.87%, highlighting that even with reduced tail risk, the portfolio can still experience noticeable drawdowns over time.

In [55]:
# CRedit

# Credit Analysis

### Defining Bond prices 

In [59]:
# Extract bond returns (for context only)
bond_returns = returns[bonds]

print(bond_returns.head())

Ticker           LQD       HYG
Date                          
2020-01-03  0.003431 -0.001020
2020-01-06 -0.003342 -0.001020
2020-01-07 -0.002573 -0.000907
2020-01-08 -0.001407  0.000908
2020-01-09  0.004776  0.002610


In [60]:
# Credit assumptions
credit_data = pd.DataFrame({
    "Asset": bonds,
    "EAD": [1_000_000, 1_200_000],   # exposure
    "PD": [0.02, 0.04],              # investment grade vs high yield
    "LGD": [0.60, 0.65]              # typical recovery assumptions
})

credit_data

,Asset,EAD,PD,LGD
0,LQD,1000000,0.02,0.60
1,HYG,1200000,0.04,0.65


### Credit Risk Measures (EL, UL, Credit VaR)
We calculate Expected Loss (EL), Unexpected Loss (UL), and Credit VaR to measure both average and extreme credit risk in the portfolio.

In [61]:
from scipy.stats import norm

confidence = 0.99
z = norm.ppf(confidence)

# Expected Loss
credit_data["EL"] = credit_data["EAD"] * credit_data["PD"] * credit_data["LGD"]

# Unexpected Loss
credit_data["UL"] = credit_data["EAD"] * credit_data["LGD"] * np.sqrt(
    credit_data["PD"] * (1 - credit_data["PD"])
)

# Credit VaR
credit_data["Credit_VaR"] = credit_data["EL"] + z * credit_data["UL"]

credit_data

,Asset,EAD,PD,LGD,EL,UL,Credit_VaR
0,LQD,1000000,0.02,0.60,12000.0,84000.00000,207413.221419
1,HYG,1200000,0.04,0.65,31200.0,152848.15995,386777.991950


# Total Credit Var
We sum the individual Credit VaR values to get the total portfolio credit risk at the 99% level.

In [62]:
total_credit_var = credit_data["Credit_VaR"].sum()

print("Total Credit VaR:", total_credit_var)

Total Credit VaR: 594191.2133694005


Interpretation of Results

The total Credit VaR is approximately 594\,191.21, which represents the portfolio’s potential loss under an extreme credit risk scenario at the 99\% confidence level. This suggests that the portfolio is exposed to substantial downside credit risk, with some exposures contributing more heavily than others.

In [63]:
final_results = pd.DataFrame({
    "Risk Type": ["Market VaR", "CVaR", "EVaR", "CDaR", "Credit VaR"],
    "Value": [VaR_95, CVaR_95, prob_evar.value, prob_cdar.value, total_credit_var]
})

print(final_results)

    Risk Type          Value
0  Market VaR       0.018514
1        CVaR       0.030375
2        EVaR       0.028086
3        CDaR       0.098684
4  Credit VaR  594191.213369


In [91]:
# Results table
results = pd.DataFrame({
    "Measure": [
        "Daily Vol",
        "VaR 95% (historical)",
        "CVaR 95% (historical)",
        "VaR 95% (Normal)",
        "CVaR 95% (Normal)",
        "Optimal CVaR",
        "Optimal EVaR",
        "Optimal CDaR",
        "Total Credit VaR (99%)"
    ],
    "Value": [
        daily_vol,
        VaR_95,
        CVaR_95,
        VaR_normal,
        CVaR_normal,
        prob_cvar.value,
        prob_evar.value,
        prob_cdar.value,
        total_credit_var
    ]
})

results

,Measure,Value
0,Daily Vol,0.013866
1,VaR 95% (historical),0.018514
2,CVaR 95% (historical),0.030375
3,VaR 95% (Normal),0.022399
4,CVaR 95% (Normal),0.028193
5,Optimal CVaR,0.013740
6,Optimal EVaR,0.028086
7,Optimal CDaR,0.098684
8,Total Credit VaR (99%),594191.213369


# Overaall interpretation

The results show that while the portfolio’s day-to-day risk is relatively moderate, the tail risk is much more significant, with losses becoming noticeably larger in extreme scenarios. The optimisation helps reduce this downside risk, especially under the CVaR measure, although more conservative measures like EVaR and CDaR show that risk is still present from different perspectives.

When credit risk is included, it becomes clear that it dominates the overall risk profile, with the total Credit VaR being much larger than the market risk measures. This suggests that the main source of risk in the portfolio comes from credit exposures rather than market fluctuations.